Libraries

In [1]:
import yfinance as yf
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

Initial setup

In [2]:
import numpy as np

tickers = pd.read_csv(
    "asx100_names.csv"
)["tickers"].tolist()
tickers.pop()
print(tickers)

data = yf.download(
    tickers,
    start="2015-01-01",
    auto_adjust=True,
    group_by="ticker",
    threads=True
)

# build master data panel
dfs = []
for ticker in tickers:
    try:
        temp = data[ticker].copy()
        temp["Ticker"] = ticker
        temp = temp.reset_index()
        dfs.append(temp)
    except:
        print(f"Missing {ticker}")

panel = pd.concat(dfs)

['A2M.AX', 'AFI.AX', 'AGL.AX', 'AIA.AX', 'ALD.AX', 'ALL.AX', 'ALQ.AX', 'ALX.AX', 'AMC.AX', 'ANN.AX', 'ANZ.AX', 'APA.AX', 'APT.AX', 'ARG.AX', 'AST.AX', 'ASX.AX', 'AWC.AX', 'AZJ.AX', 'BEN.AX', 'BHP.AX', 'BLD.AX', 'BOQ.AX', 'BSL.AX', 'BXB.AX', 'CBA.AX', 'CCL.AX', 'CHC.AX', 'CIM.AX', 'COH.AX', 'COL.AX', 'CPU.AX', 'CSL.AX', 'CWN.AX', 'CWY.AX', 'DMP.AX', 'DXS.AX', 'EVN.AX', 'FBU.AX', 'FMG.AX', 'FPH.AX', 'GMG.AX', 'GPT.AX', 'HVN.AX', 'IAG.AX', 'IEL.AX', 'IGO.AX', 'IPL.AX', 'JBH.AX', 'JHX.AX', 'LLC.AX', 'MCY.AX', 'MEZ.AX', 'MFG.AX', 'MGOC.AX', 'MGR.AX', 'MIN.AX', 'MPL.AX', 'MQG.AX', 'NAB.AX', 'NCM.AX', 'NST.AX', 'NXT.AX', 'ORG.AX', 'ORI.AX', 'OSH.AX', 'OZL.AX', 'PMGOLD.AX', 'QAN.AX', 'QBE.AX', 'QUB.AX', 'REA.AX', 'REH.AX', 'RHC.AX', 'RIO.AX', 'RMD.AX', 'S32.AX', 'SCG.AX', 'SEK.AX', 'SGP.AX', 'SHL.AX', 'SOL.AX', 'SPK.AX', 'STO.AX', 'SUN.AX', 'SVW.AX', 'SYD.AX', 'TAH.AX', 'TCL.AX', 'TLS.AX', 'TPG.AX', 'TWE.AX', 'VAS.AX', 'VCX.AX', 'WBC.AX', 'WES.AX', 'WOR.AX', 'WOW.AX', 'WPL.AX', 'WTC.AX', 'XRO.

[*******************   39%                       ]  39 of 100 completed$AWC.AX: possibly delisted; no timezone found
[**********************50%                       ]  50 of 100 completed$APT.AX: possibly delisted; no timezone found
[**********************51%                       ]  51 of 100 completed$BLD.AX: possibly delisted; no timezone found
[**********************52%                       ]  52 of 100 completed$CWN.AX: possibly delisted; no timezone found
[**********************53%                       ]  53 of 100 completed$IPL.AX: possibly delisted; no timezone found
[**********************54%*                      ]  54 of 100 completed$OSH.AX: possibly delisted; no timezone found
[**********************64%******                 ]  64 of 100 completed$SYD.AX: possibly delisted; no timezone found
[**********************64%******                 ]  64 of 100 completed$WPL.AX: possibly delisted; no timezone found
[**********************69%********               ]  69 of 100 co

Feature Engineering:

In [3]:
#Momentum and reversal features
panel["mom_1m"] = panel.groupby("Ticker")["Close"].pct_change(21)
panel["mom_3m"] = panel.groupby("Ticker")["Close"].pct_change(63)
panel["mom_6m"] = panel.groupby("Ticker")["Close"].pct_change(126)
panel["mom_12m"] = panel.groupby("Ticker")["Close"].pct_change(252)
panel["reversal_1m"] = -panel["mom_1m"]

#Daily Returns
panel["daily_return"] = panel.groupby("Ticker")["Close"].pct_change()

# Add market returns
market = yf.download("^AXJO", start="2015-01-01", auto_adjust=True)
market.columns = market.columns.get_level_values(0)
market = market.reset_index()
market["market_return"] = market["Close"].pct_change()
panel = panel.merge(
    market[["Date", "market_return"]],
    on="Date",
    how="left"
)
print(panel[["Date", "Ticker", "market_return"]].head())

# Volatility features (annualized)
panel["vol_20"] = (
    panel.groupby("Ticker")["daily_return"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
) * np.sqrt(252)

panel["vol_60"] = (
    panel.groupby("Ticker")["daily_return"]
    .rolling(60)
    .std()
    .reset_index(level=0, drop=True)
) * np.sqrt(252)

#Ratio: this is short term to long term volatility
panel["vol_ratio"] = (panel["vol_20"] /panel["vol_60"])

# downside volatility
panel["negative_return"] = panel["daily_return"].clip(upper=0)
panel["downside_vol"] = (
    panel.groupby("Ticker")["negative_return"]
    .transform(lambda x: x.rolling(60).std())
) * np.sqrt(252)


#Liquidity
panel["dollar_volume"] = (panel["Close"] * panel["Volume"])

avg_volume = (
    panel.groupby("Ticker")["Volume"]
    .rolling(20)
    .mean()
    .reset_index(level=0, drop=True)
)

panel["volume_shock"] = (panel["Volume"]/avg_volume)

# Illiquidity
panel["amihud"] = (
    panel["daily_return"].abs()/ panel["dollar_volume"]
)

# Beta features (short, medium and long term)
def calculate_beta(group, window):
    covariance = (
        group["daily_return"]
        .rolling(window)
        .cov(group["market_return"])
    )
    market_variance = (
        group["market_return"]
        .rolling(window)
        .var()
    )
    return covariance / market_variance

panel["beta_252"] = (
    panel.groupby("Ticker", group_keys=False)
    .apply(calculate_beta, window=252)
)

panel["beta_126"] = (
    panel.groupby("Ticker", group_keys=False)
    .apply(calculate_beta, window=126)
)  

panel["beta_63"] = (
    panel.groupby("Ticker", group_keys=False)
    .apply(calculate_beta, window=63)
)

# risk features
def calculate_idio_volatility(residualwindow):
    panel[f"expected_return_{residualwindow}"] = (
        panel[f"beta_{residualwindow}"] * panel["market_return"]
    )
    panel[f"residual_return_{residualwindow}"] = (
        panel["daily_return"] - panel[f"expected_return_{residualwindow}"]
    )
    return (
        panel.groupby("Ticker")[f"residual_return_{residualwindow}"]
        .rolling(60)
        .std()
        .reset_index(level=0, drop=True)
    )
panel["idio_vol_252"] = calculate_idio_volatility(252)
panel["idio_vol_126"] = calculate_idio_volatility(126)
panel["idio_vol_63"] = calculate_idio_volatility(63)

panel = panel.dropna()

# TO ADD:
# - Market regime variables, macro variables
# - interaction variables between features (e.g., momentum * volatility, etc.)

# cross-sectional standardization:
feature_cols = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "mom_12m",
    "reversal_1m",
    "vol_20",
    "vol_60",
    "vol_ratio",
    "downside_vol",
    "volume_shock",
    "dollar_volume", #double check
    "amihud",
    "beta_252",
    "beta_126",
    "beta_63",
    "idio_vol_252",
    "idio_vol_126",
    "idio_vol_63"
]

for feature in feature_cols:
    panel[f"{feature}_z"] = (
        panel.groupby("Date")[feature]
        .transform(
            lambda x:
            (x - x.mean()) / x.std()
        )
    )  

[*********************100%***********************]  1 of 1 completed


Price       Date  Ticker  market_return
0     2015-01-02  A2M.AX            NaN
1     2015-01-05  A2M.AX       0.002649
2     2015-01-06  A2M.AX      -0.015687
3     2015-01-07  A2M.AX      -0.002088
4     2015-01-08  A2M.AX       0.005211


Train/test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb


